# Risk Model Retraining\n\nRetrains `artifacts/xgb.pkl` using `data/project_details.xlsx`.

In [ ]:
from pathlib import Path\nfrom datetime import datetime\nimport pickle\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import classification_report, accuracy_score, f1_score\nfrom xgboost import XGBClassifier\n\nbase = Path('data/project_details.xlsx')\nart = Path('artifacts')\nart.mkdir(exist_ok=True)\n\ndf = pd.read_excel(base)\nfeatures = ['Domain','Mobile','Desktop','Web','IoT','Expected Team Size','Expected Budget']\ntarget = 'Risk'\nX = df[features].copy()\ny = df[target].astype(int) - 1\n\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.2, random_state=42, stratify=y\n)\n\nmodel = XGBClassifier(\n    n_estimators=400,\n    max_depth=6,\n    learning_rate=0.05,\n    subsample=0.9,\n    colsample_bytree=0.9,\n    objective='multi:softmax',\n    num_class=3,\n    eval_metric='mlogloss',\n    random_state=42,\n    n_jobs=4,\n)\nmodel.fit(X_train, y_train)\n\npred = model.predict(X_test)\nprint('accuracy', round(accuracy_score(y_test, pred), 4))\nprint('macro_f1', round(f1_score(y_test, pred, average='macro'), 4))\nprint(classification_report(y_test, pred, digits=4))

In [ ]:
old = art / 'xgb.pkl'\nif old.exists():\n    ts = datetime.now().strftime('%Y%m%d_%H%M%S')\n    bak = art / f'xgb.pkl.bak_{ts}'\n    bak.write_bytes(old.read_bytes())\n    print('backup:', bak)\n\nwith open(old, 'wb') as f:\n    pickle.dump(model, f)\nprint('saved model:', old)\n\nout = art / 'xgb_risk_metrics.txt'\nout.write_text(\n    'features: ' + ', '.join(features) + '\\n' +\n    f'accuracy: {accuracy_score(y_test, pred):.6f}\\n' +\n    f'macro_f1: {f1_score(y_test, pred, average=\"macro\"):.6f}\\n'\n)\nprint('saved metrics:', out)